<a href="https://colab.research.google.com/github/Yuylam/triple-a/blob/main/p1/maukerja_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cleaning Raw Data

## Load data

In [ ]:
import polars as pl

# Load dataset
input_path = "../data/maukerja_compiled_with_description.csv"

# Drop duplicate & unnecessary column
col_used = ["job_id", "salary", "location", "scraped_location", "url", "title", "company", "scraped_description"]
strict_schema = {col: pl.String for col in col_used}

# Read data
df = pl.read_csv(
    input_path,
    columns=col_used,
    schema_overrides=strict_schema,
    infer_schema_length=0,
    ignore_errors=True,
    truncate_ragged_lines=True
)

print(f"Success! Loaded {df.height:,} records.")

## Remove duplicate

In [ ]:
# Remove duplicate
df_clean = (
    df.filter(pl.col("job_id").is_not_null() & (pl.col("job_id") != "N/A"))
    .unique(subset=["job_id"], keep="first")
)

# Fill nulls globally
df_clean = df_clean.with_columns([pl.col(col).fill_null("") for col in df_clean.columns])

# text trimming
string_columns = [name for name, dtype in zip(df_clean.columns, df_clean.dtypes) if dtype == pl.String]
df_clean = df_clean.with_columns([
    pl.col(col).str.strip_chars().str.replace_all(r"\s+", " ") for col in string_columns
])

print(df_clean.shape[0])

## Cleaning Salary Column

In [ ]:
# Normalizing salary
df_clean = df_clean.with_columns(
    pl.col("salary")
    .str.to_lowercase()
    .str.replace_all(",", "")
    .str.replace_all(r"[^0-9\-]", "")
    .str.strip_chars()
    .alias("sal_raw")
)

# Extract minimum and maximum salary
df_clean = df_clean.with_columns([
    pl.col("sal_raw").str.extract(r"(\d+)-(\d+)", 1).cast(pl.Float64, strict=False).alias("s_min"),
    pl.col("sal_raw").str.extract(r"(\d+)-(\d+)", 2).cast(pl.Float64, strict=False).alias("s_max")
])

df_clean = df_clean.with_columns([
    pl.when(pl.col("s_min").is_null())
    .then(pl.col("sal_raw").str.extract(r"(\d+)", 1).cast(pl.Float64, strict=False))
    .otherwise(pl.col("s_min"))
    .alias("s_min")
])
df_clean = df_clean.with_columns([
    pl.when(pl.col("s_max").is_null()).then(pl.col("s_min")).otherwise(pl.col("s_max")).alias("s_max")
])

# Standardize currency
df_clean = df_clean.with_columns([
    pl.when(pl.col("salary").str.to_lowercase().str.contains(r"sgd|s\$") |
            pl.col("location").str.to_lowercase().str.contains("singapore|singapura") |
            pl.col("scraped_location").str.to_lowercase().str.contains("singapore|singapura"))
    .then(pl.col("s_min") * 3.1)
    .otherwise(pl.col("s_min"))
    .alias("salary_min_numeric"),

    pl.when(pl.col("salary").str.to_lowercase().str.contains(r"sgd|s\$") |
            pl.col("location").str.to_lowercase().str.contains("singapore|singapura") |
            pl.col("scraped_location").str.to_lowercase().str.contains("singapore|singapura"))
    .then(pl.col("s_max") * 3.1)
    .otherwise(pl.col("s_max"))
    .alias("salary_max_numeric"),
])

# Standardize rate
df_clean = df_clean.with_columns([
    pl.when(pl.col("salary").str.to_lowercase().str.contains("hour|jam|sejam")).then(pl.col("salary_min_numeric") * 8 * 22)
    .when(pl.col("salary").str.to_lowercase().str.contains("day|hari|sehari")).then(pl.col("salary_min_numeric") * 22)
    .otherwise(pl.col("salary_min_numeric")).alias("salary_min_numeric"),

    pl.when(pl.col("salary").str.to_lowercase().str.contains("hour|jam|sejam")).then(pl.col("salary_max_numeric") * 8 * 22)
    .when(pl.col("salary").str.to_lowercase().str.contains("day|hari|sehari")).then(pl.col("salary_max_numeric") * 22)
    .otherwise(pl.col("salary_max_numeric")).alias("salary_max_numeric"),
])

# Round salary value
df_clean = df_clean.with_columns([
    pl.when((pl.col("salary_min_numeric") % 1) == 0)
    .then(pl.col("salary_min_numeric").cast(pl.Int64).cast(pl.String))
    .otherwise(pl.col("salary_min_numeric").round(2).cast(pl.String))
    .alias("salary_min"),

    pl.when((pl.col("salary_max_numeric") % 1) == 0)
    .then(pl.col("salary_max_numeric").cast(pl.Int64).cast(pl.String))
    .otherwise(pl.col("salary_max_numeric").round(2).cast(pl.String))
    .alias("salary_max")
])

# Drop unwanted columns
df_clean = df_clean.drop(["salary", "sal_raw", "s_min", "s_max", "salary_min_numeric", "salary_max_numeric"])


## Regional State Standardization

In [ ]:
# Combine text location
df_clean = df_clean.with_columns(
    (pl.col("location").str.to_lowercase() + pl.lit(" ") + pl.col("scraped_location").str.to_lowercase()).alias("loc_combined")
)

# Extract state
df_clean = df_clean.with_columns(
    pl.when(pl.col("loc_combined").str.contains("singapore|singapura")).then(pl.lit("International"))
    .when(pl.col("loc_combined").str.contains("philippines|philipines|manila|filipina")).then(pl.lit("International"))
    .when(pl.col("loc_combined").str.contains("indonesia|jakarta|surabaya|medan|indo\b")).then(pl.lit("International"))
    .when(pl.col("loc_combined").str.contains("united states|united state|\busa\b|america|\bus\b")).then(pl.lit("International"))

    .when(pl.col("loc_combined").str.contains(
        r"kuala lumpur|putrajaya|\bkl\b|federal territory|bukit bintang|kepong|sungai besi|sentul|bangsar|brickfield|bandar menjalara|chan sow lin|taman desa|setapak|segambut|klcc|kuchai lama|wangsa maju|cheras kl|titiwangsa|pantai dalam"
    )).then(pl.lit("Kuala Lumpur"))

    .when(pl.col("loc_combined").str.contains(
        r"selangor|shah alam|petaling jaya|\bpj\b|subang|klang|cyberjaya|puchong|cheras|bangi|serdang|rawang|kajang|gombak|ampang|sepang|damansara|sunway|elmina|semenyih|banting|kuala selangor"
    )).then(pl.lit("Selangor"))

    .when(pl.col("loc_combined").str.contains(
        r"johor|jb|skudai|pasir gudang|batu pahat|muar|kluang|segaia|kulai|senai|pontian|mersing|kota tinggi"
    )).then(pl.lit("Johor"))

    .when(pl.col("loc_combined").str.contains(
        r"penang|pulau pinang|butterworth|perai|georgetown|george town|bayan lepas|seberang jaya|kepala batas"
    )).then(pl.lit("Penang"))

    .when(pl.col("loc_combined").str.contains(
        r"kedah|alor setar|sungai petani|sp|kulim|langkawi|jitra|baling|kota kuala muda|kuala muda"
    )).then(pl.lit("Kedah"))

    .when(pl.col("loc_combined").str.contains("perak|ipoh|taiping|manjung|sitiawan|teluk intan|batu gajah|kampar|tapah|kuala kangsar|tanjung malim")).then(pl.lit("Perak"))
    .when(pl.col("loc_combined").str.contains("kelantan|kota bharu|kb|pasir mas|tumpat|tanah merah|machang|kuala krai")).then(pl.lit("Kelantan"))
    .when(pl.col("loc_combined").str.contains("terengganu|kuala terengganu|kt|kemaman|dungun|marang|besut")).then(pl.lit("Terengganu"))
    .when(pl.col("loc_combined").str.contains("pahang|kuantan|temerloh|bentong|mentakab|pekan|raub|jerantut|cameron highlands")).then(pl.lit("Pahang"))
    .when(pl.col("loc_combined").str.contains("negeri sembilan|seremban|nilai|port dickson|pd|senawang|rembau|tampin")).then(pl.lit("Negeri Sembilan"))
    .when(pl.col("loc_combined").str.contains("melaka|malacca|alor gajah|jasin")).then(pl.lit("Melaka"))
    .when(pl.col("loc_combined").str.contains("perlis|kangar|arau|kuala perlis")).then(pl.lit("Perlis"))
    .when(pl.col("loc_combined").str.contains("sarawak|kuching|miri|sibu|bintulu|samarahan|sarikei")).then(pl.lit("Sarawak"))
    .when(pl.col("loc_combined").str.contains("sabah|kota kinabalu|kk|sandakan|tawau|lahad datu|penampang")).then(pl.lit("Sabah"))
    .when(pl.col("loc_combined").str.contains("labuan")).then(pl.lit("Labuan"))

    .when(pl.col("loc_combined").str.strip_chars() == "malaysia").then(pl.lit("Nationwide"))
    .when(pl.col("loc_combined").str.contains("remote|work from home|wfh|bekerja dari rumah")).then(pl.lit("Remote"))
    .otherwise(pl.lit("Nationwide"))
    .alias("state")
)

# Extract country
df_clean = df_clean.with_columns(
    pl.when(pl.col("loc_combined").str.contains("singapore|singapura")).then(pl.lit("Singapore"))
    .when(pl.col("loc_combined").str.contains("philippines|philipines|manila|filipina")).then(pl.lit("Philippines"))
    .when(pl.col("loc_combined").str.contains("indonesia|jakarta|surabaya|medan|indo\b")).then(pl.lit("Indonesia"))
    .when(pl.col("loc_combined").str.contains("united states|united state|\busa\b|america|\bus\b")).then(pl.lit("United States"))
    .when(pl.col("state") == "Remote").then(pl.lit("Remote/International"))
    .otherwise(pl.lit("Malaysia"))
    .alias("country")
)

# Drop unwanted column
df_clean = df_clean.drop(["location", "scraped_location", "loc_combined"])


## Rename Column

In [ ]:
df_clean = df_clean.rename({"scraped_description": "description"})

## Save file

In [ ]:
df_clean.write_csv("../data/maukerja_cleaned.csv")
df_clean.height

In [ ]:
# CSV TO GZIP
import pandas as pd 
filename = "../data/maukerja_cleaned"  
df = pd.read_csv(f"{filename}.csv")
df.to_csv(f"{filename}.gz", index=False, compression="gzip")

In [ ]:
# !gzip -f maukerja_cleaned_updated.csv

In [ ]:
# # download file
# from google.colab import files
# files.download("maukerja_cleaned_updated.csv.gz")